**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**01 | Original Dataset Audit and Trial Registry**

**Research question:** Are the original six-activity Essex source files authentic, participant-linked and traceable to unique movement records and original trial identifiers?

**Published source:** Hollinger et al. (2026), *Scientific Data*, [DOI 10.1038/s41597-026-07429-3](https://doi.org/10.1038/s41597-026-07429-3); [Figshare 30225739, version 13](https://doi.org/10.6084/m9.figshare.30225739.v13). This is the original full MATLAB dataset, not the separate 444-record BoneNet derivative.

**Scope and reproducibility:** Original record identifiers are preserved. Physical IMU/FE units, decoded strain waveforms, temporal alignment, clinical use and subject-independent regression are separate research questions. Original Notebooks 01/02 are preserved in Git history; their results are not overwritten; new files are saved directly in `output/`. Five names overlapping original Stage 01 files use the distinct `01_dataset_*` prefix to avoid accidental replacement.


**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content
/content/human-movement-intelligence


**01. Imports and Environment**


In [ ]:
# ======================== 01. Imports and Environment ========================

from python_modules.core.colab_setup import setup_colab

root = setup_colab("biomechanical_load_estimation")
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans",
    title_size=12.5,
    title_weight="normal",
    label_size=11.5,
    label_weight="normal",
    tick_size=11,
    legend_size=10,
    figure_dpi=100,
    figure_size=(9, 4),
)
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID,
    ARTICLE_VERSION,
    load_source_manifest,
    find_published_file,
    ensure_source_file,
    ensure_matlab_archive,
    inspect_matlab_archive,
    extract_matlab_files,
    inspect_subject_mat,
)
from essex_record_audit import (
    load_participant_ids,
    audit_mat_records,
    summarise_trial_evidence,
)

raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = root / "biomechanical_load_estimation" / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Figshare article:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("Research outputs saved directly to:", output_dir)

**02. Dataset Source Files**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 02. Dataset Source Files

article, source_files = load_source_manifest(output_dir)
archive_file = find_published_file(source_files, "MATLAB.zip")
source_table = pd.DataFrame(
    [
        {
            "file_id": item["id"],
            "filename": item["name"],
            "size_bytes": item["size"],
            "download_url": item["download_url"],
        }
        for item in source_files
    ]
)
source_table.to_csv(output_dir / "01_dataset_figshare_file_manifest.csv", index=False)
display(source_table)
print("Source:", ARTICLE_ID, "version", ARTICLE_VERSION)

**03. Dataset Archive**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 03. Download and Open the Dataset Archive

archive_path = ensure_matlab_archive(raw_dir)
zip_records = inspect_matlab_archive(archive_path)
archive_table = pd.DataFrame(zip_records)
archive_table.to_csv(
    output_dir / "01_dataset_matlab_archive_inventory.csv", index=False
)
display(archive_table.head(10))
print("MATLAB files:", archive_table["is_matlab_file"].sum())

**04. Original Participant File Inventory**

Extract the 40 published participant MAT files and preserve their authentic, nonconsecutive identifiers. Check source file integrity and inventory.


In [ ]:
# ================== 04. Original Participant File Inventory ==================

mat_paths = extract_matlab_files(archive_path, mat_dir, zip_records)
participant_ids = [path.stem.lower() for path in mat_paths]
participant_table = pd.DataFrame(
    {
        "source_file": [path.name for path in mat_paths],
        "subject_id": participant_ids,
        "file_size_bytes": [path.stat().st_size for path in mat_paths],
    }
)
display(participant_table)
participant_table.to_csv(
    output_dir / "01_dataset_matlab_participant_inventory.csv", index=False
)
print("Independent source files:", len(participant_table))
print("IDs are observed from filenames, not generated by range.")

**05. Original MATLAB Source Schema**

In [ ]:
# ===================== 05. Original MATLAB Source Schema =====================

audit_rows = []
for source_path in mat_paths:
    audit_rows.append(inspect_subject_mat(source_path))
audit_table = pd.DataFrame(audit_rows).sort_values("subject_id").reset_index(drop=True)
display(
    audit_table[
        [
            "subject_id",
            "record_count",
            "source_data_shape",
            "first_record_type",
            "field_names_first_record",
        ]
    ].head(12)
)
audit_table.to_csv(
    output_dir / "01_dataset_matlab_record_schema_audit.csv", index=False
)
print("Participants audited:", len(audit_table))
print("Total MATLAB Data records:", int(audit_table["record_count"].sum()))
print("Record counts by participant:")
display(audit_table[["subject_id", "record_count"]])

**06. Original FE Field Representation Census**

Inspect e1e3 source fields per participant and classify source-empty, opaque and numeric candidate representations. An opaque MATLAB object is not equivalent to a validated FE waveform.


In [ ]:
# ================ 06. Original FE Field Representation Census ================

status_cols = [
    "numeric_candidate",
    "empty",
    "missing_field",
    "nonfinite_numeric",
    "matlab_object_unresolved",
    "non_numeric_unresolved",
]
target_table = audit_table[["subject_id", "record_count", *status_cols]].copy()
target_table.to_csv(
    output_dir / "01_dataset_femoral_strain_representation.csv", index=False
)
display(target_table)
print("FE representation totals:")
display(target_table[status_cols].sum().rename("records").to_frame())

**07. Participant Workbook**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 07. Participant Workbook

workbook_item = find_published_file(source_files, "SubCharacteristics.xlsx")
workbook_path = ensure_source_file(workbook_item, raw_dir)
workbook_summary, subjects = load_participant_ids(workbook_path)
display(pd.DataFrame([workbook_summary]))
subjects.to_csv(output_dir / "01_workbook_subject_ids.csv", index=False)
display(subjects.head())

**08. Original Trial Identity and Record Registry**

Audit all 1,897 Data elements with original UpdatedTrialID, task, subtask, participant and file-local index; preserve unique source keys and check participant/trial duplication.


In [ ]:
# ============== 08. Original Trial Identity and Record Registry ==============

records = audit_mat_records(mat_paths)
trial_summary = summarise_trial_evidence(records)
display(
    records[
        [
            "subject_id",
            "local_record_index",
            "task",
            "subtask",
            "updated_trial_id",
            "imu_representation",
            "fe_representation",
        ]
    ].head(18)
)
print("Audited source records:", trial_summary["source_records"])
print("Participants:", trial_summary["source_participants"])
print("Readable UpdatedTrialID fields:", trial_summary["readable_updated_trial_ids"])
records.to_csv(output_dir / "01_original_record_registry.csv", index=False)
print("Duplicate participant/trial IDs:", trial_summary["duplicate_subject_trial_ids"])
print("No inferred source-trial identifiers were created.")

**09. Original IMU and FE Field Representation**

Describe the original IMU and FE representation at the record level; distinguish empty FE source fields from opaque objects. Do not equate structural representations with numeric signal validation.


In [ ]:
# =============== 09. Original IMU and FE Field Representation ================

coverage = (
    records.groupby(["subject_id", "imu_representation", "fe_representation"])
    .size()
    .rename("records")
    .reset_index()
)
coverage.to_csv(output_dir / "01_modality_representation.csv", index=False)
display(coverage.head(18))
encoding = pd.DataFrame(
    [
        {"modality": "IMU", "representation": k, "records": int(v)}
        for k, v in trial_summary["imu_encoding_counts"].items()
    ]
    + [
        {"modality": "FE", "representation": k, "records": int(v)}
        for k, v in trial_summary["fe_encoding_counts"].items()
    ]
)
display(encoding)
encoding.to_csv(output_dir / "01_modality_encoding_summary.csv", index=False)

**10. Original Activity and FE Field Coverage**

Summarise all original activity categories and the distribution of FE source representations. This is descriptive coverage, not a model-performance assessment.


In [ ]:
# ================ 10. Original Activity and FE Field Coverage ================

activity_counts = (
    records.groupby("task", dropna=False).size().sort_values(ascending=False)
)
display(activity_counts.rename("records").to_frame())
activity_coverage = (
    records.groupby(["task", "fe_representation"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
activity_coverage.to_csv(output_dir / "01_activity_fe_coverage.csv", index=False)
display(activity_coverage)

**11. Consolidated Research Coverage Figure**

Visualise participant, activity and FE source-field coverage in one figure, avoiding duplicate panels while preserving original figures as historical evidence.


In [ ]:
# ================= 11. Consolidated Research Coverage Figure =================

subject_counts = records.groupby("subject_id").size().sort_index()
fe_counts = records["fe_representation"].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
positions = list(range(len(subject_counts)))
axes[0].bar(positions, subject_counts.values)
axes[0].set_xticks(positions[::4])
axes[0].set_xticklabels(subject_counts.index[::4], rotation=65)
axes[0].set_title("Original source records by participant")
axes[0].set_ylabel("Source records")
axes[1].barh(activity_counts.index, activity_counts.values)
axes[1].invert_yaxis()
axes[1].set_title("Original activity distribution")
axes[1].set_xlabel("Source records")
axes[2].barh(fe_counts.index, fe_counts.values)
axes[2].invert_yaxis()
axes[2].set_title("Source FE representations")
axes[2].set_xlabel("Source records")
save_figure(fig, output_dir / "01_combined_source_coverage.png")
plt.show()

**12. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 12. Results Summary and Output Download

display(
    pd.DataFrame(
        [
            {
                "participants": records["subject_id"].nunique(),
                "trials": len(records),
                "activities": records["task"].nunique(),
                "missing_activity": int(records["task"].isna().sum()),
                "duplicate_trial_keys": int(
                    records["local_record_key"].duplicated().sum()
                ),
                "empty_fe_records": int(records["fe_representation"].eq("empty").sum()),
            }
        ]
    )
)
print("Current trial registry: 01_original_record_registry.csv")
download_outputs_from_colab(folder="biomechanical_load_estimation/output", stage="01")

**13. Research Interpretation and Next Stage Boundary**

**Scientific limitations:** Verified source integrity and original field representation are not the same as numerical FE waveform recovery, physical unit verification, time alignment, valid independent Test splitting or clinical generalisation.